# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up a persistent SQLite vector store with sqlite-vec
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies
- `tip_hvac_efficiency.txt` - Heating, cooling, thermostat settings, and maintenance
- `tip_home_automation.txt` - Sensors, smart plugs, and tariff-based schedules
- `tip_residential_renewables.txt` - Rooftop solar and using surplus generation
- `tip_seasonal_energy.txt` - Preparing the home for hot and cold weather
- `tip_battery_storage.txt` - Charging, discharging, usable capacity, and backup reserve

## 1. Import Required Libraries


In [1]:
import os
from dotenv import load_dotenv

from tools import (
    load_energy_tip_documents,
    open_energy_tips_index,
    split_energy_tip_documents,
)

In [2]:
load_dotenv()
if not os.getenv("VOCAREUM_API_KEY"):
    raise RuntimeError("Set VOCAREUM_API_KEY before building or testing the RAG index.")
print("Vocareum configuration is available.")

Vocareum configuration is available.


## 2. Load and Process Documents


In [3]:
# Load all plain-text guides and keep their source paths for citations.
documents = load_energy_tip_documents()
for document in documents:
    print(f"Loaded {document.metadata['source']}")
print(f"Total documents loaded: {len(documents)}")

Loaded data/documents/tip_battery_storage.txt
Loaded data/documents/tip_device_best_practices.txt
Loaded data/documents/tip_energy_savings.txt
Loaded data/documents/tip_home_automation.txt
Loaded data/documents/tip_hvac_efficiency.txt
Loaded data/documents/tip_residential_renewables.txt
Loaded data/documents/tip_seasonal_energy.txt
Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split the guides into overlapping chunks using the same settings as the search tool.
splits = split_energy_tip_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 14 chunks

Sample chunk (first 200 characters):
Home battery storage for solar power and backup

Charging and discharging a home battery
A battery can store excess solar generation during the day and supply electricity after sunset. Charge from sur...


## 4. Create Vector Store


In [5]:
# Use the same SQLite index and embedding provider as the agent's search tool.
vectorstore = open_energy_tips_index()
chunk_count = vectorstore.index(splits)

print(f"Vector store: {vectorstore.path.as_posix()}")
print(f"Total vectors stored: {chunk_count}")

Vector store: data/energy_tips.sqlite3
Total vectors stored: 14


## 5. Test the RAG Pipeline


In [6]:
loaded_sources = {document.metadata["source"] for document in documents}
indexed_sources = vectorstore.sources()
assert indexed_sources == loaded_sources, (loaded_sources - indexed_sources)

print(f"Verified all {len(indexed_sources)} guides in the SQLite index.")
query = "How much electricity can a home battery reserve for an outage?"
print(f"\nDirect vector search: {query}")
for rank, document in enumerate(vectorstore.search(query, limit=2), start=1):
    print(f"{rank}. {document.metadata['source']}")
    print(f"   {document.page_content[:160]}...")

Verified all 7 guides in the SQLite index.

Direct vector search: How much electricity can a home battery reserve for an outage?
1. data/documents/tip_battery_storage.txt
   Home battery storage for solar power and backup

Charging and discharging a home battery
A battery can store excess solar generation during the day and supply e...
2. data/documents/tip_battery_storage.txt
   Backup reserve and supported loads
Set the reserve according to how much energy the household wants to retain for an outage. A higher reserve leaves less energy...


## 6. Test the Search Tool


In [7]:
from pathlib import Path

from tools import search_energy_tips

# Each query should recover its expected guide among the top three sources.
test_cases = [
    (
        "How do HVAC filter changes and blocked return grilles affect air conditioning airflow?",
        "tip_hvac_efficiency.txt",
    ),
    (
        "How can occupancy sensors and smart plugs help automate energy use under time-of-use rates?",
        "tip_home_automation.txt",
    ),
    (
        "When should I run my dishwasher to use surplus rooftop solar instead of exporting it?",
        "tip_residential_renewables.txt",
    ),
    (
        "What seasonal energy-saving steps should I take before a summer heat wave?",
        "tip_seasonal_energy.txt",
    ),
    (
        "How can a home battery store midday solar until after sunset while keeping an outage reserve?",
        "tip_battery_storage.txt",
    ),
]

print("=== Testing search_energy_tips Tool ===")
for query, expected_source in test_cases:
    result = search_energy_tips.invoke({"query": query, "max_results": 3})
    assert "error" not in result, f"{query}: {result['error']}"

    sources = [Path(tip["source"]).name for tip in result["tips"]]
    assert expected_source in sources, f"{expected_source} missing from {sources}"

    rank = sources.index(expected_source) + 1
    match = result["tips"][rank - 1]
    print(f"\nQuery: {query}")
    print(f"  Matched guide at rank {rank}: {match['source']}")
    print(f"  {match['content'][:160]}...")

print(f"\nPassed all {len(test_cases)} guide retrieval checks.")

=== Testing search_energy_tips Tool ===

Query: How do HVAC filter changes and blocked return grilles affect air conditioning airflow?
  Matched guide at rank 1: data/documents/tip_hvac_efficiency.txt
  Home heating, ventilation, and air conditioning (HVAC)

Filters and airflow
Check the air filter on the schedule in the equipment manual and clean or replace it...

Query: How can occupancy sensors and smart plugs help automate energy use under time-of-use rates?
  Matched guide at rank 1: data/documents/tip_home_automation.txt
  Home automation for energy management

Occupancy sensors and smart plugs
Use occupancy sensors to turn off lights in rooms that stay empty. Set a delay or manua...

Query: When should I run my dishwasher to use surplus rooftop solar instead of exporting it?
  Matched guide at rank 1: data/documents/tip_residential_renewables.txt
  Residential renewable energy and rooftop solar

Rooftop photovoltaic generation
Solar photovoltaic panels produce electricity during